<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Aim
The purpose of this notebook is present how AITune stores and loads checkpoints.

Basically, storing and loading tuned models or pipelines consists of two parts:
- checkpointing mechanism - which uses torch `state_dict` to dump and load weights
- storage mechanism - which is used to write the resulting artifacts to disk.

This notebook presents storage mechanism. The checkpointing is presented in `saving_loading_checkpoints.ipynb` notebook.

In [ ]:
%%capture
%cd ..

In [ ]:
import os
from logging import basicConfig

import torch
import torch.nn as nn
from aitune.torch import LocalTorchStorage
from aitune.torch.backend import TorchInductorJitBackend, TensorRTBackend
from aitune.torch import save, tune, LocalTorchStorage, OneBackendStrategy, Module

In [ ]:
log_level = os.environ.get("AITUNE_LOG_LEVEL", "INFO")
basicConfig(level=log_level, format="%(asctime)s - %(levelname)s - %(message)s", force=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Introduction

First let's create a fake `state_dict` which will imitate real torch module state dict.

In [ ]:
state_dict = {
    "layer" : torch.randn(2, 2),
}

state_dict

Let's save a fake `state_dict` to a checkpoint.

In [ ]:
storage = LocalTorchStorage(base_folder="notebooks/checkpoints", remove_checkpoint_after_tune=True)
storage.save("demo-storage.ait", state_dict)

Let's see stored artifacts.

In [ ]:
!find notebooks/checkpoints/demo-storage[._]* -type f

There are two files:
- the checkpoints itself
- the sha sums file

In [ ]:
!cat notebooks/checkpoints/demo-storage_sha256_sums.txt

The `demo-storage.ait` checkpoint is actually a zip archive with consists of all artifacts. When we do `load` for the first time the checkpoint is 
decompressed.


Let's try to `load` the checkpoint.

In [ ]:
loaded_state_dict = storage.load("demo-storage.ait")
loaded_state_dict

Let's see resulting folder structure.

In [ ]:
!tree -L 1 notebooks/checkpoints/demo-storage/

On the first load the sha sums are verified. On subsequent loads, to speed up the process, 
the original sha sums file is compared against the one in the decompressed folder. If they match checkpoint is loaded from decompressed folder and sums are not verified. However if the differ, the checkpoint is re-decompressed and again hash sums are verified.

# Storage with tuning

Let's create simple model and tune it.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
net = SimpleNet(5, 10, 2).to(device)
tune_data = torch.randn(5, device=device)

Let's wrap modules.

In [ ]:
net.fc1 = Module(net.fc1, "fc1", strategy=OneBackendStrategy(backend=TorchInductorJitBackend()))
net.fc2 = Module(net.fc2, "fc2", strategy=OneBackendStrategy(backend=TensorRTBackend()))
net.fc3 = Module(net.fc3, "fc3", strategy=OneBackendStrategy(backend=TensorRTBackend()))

Let's tune the model.

In [ ]:
tune(net, tune_data, batch_sizes=[1, 2], dry_run=False)

Let's save it.

In [ ]:
storage = LocalTorchStorage(
    base_folder="notebooks/checkpoints",
    remove_checkpoint_after_tune=False, # this flag will leave uncompressed checkpoint after tuning
)
save(net, "demo-tuned.ait", storage=storage)

Let's inspect the checkpoint folder.

In [ ]:
!tree -L 1 notebooks/checkpoints/demo-tuned/

It contains:
- sha sums file
- `state_dict.pt` which will have torch module weights
- two `model.plan` files - since we have two modules tuned with `TensorRTBackend`

If you would like to store checkpoints in a cloud, you can override LocalTorchStorage with a custom storage.